# ASTRAnet quickstart

End-to-end tour of the pipeline: load the config, inspect the trained ensemble's
features, run the LUNA anomaly layer, and look at the headline results.

**Prerequisites** — datasets under `data/` (see `data/README.md`) and either
trained seeds (`python pipelines/train_seeds.py`) or the released checkpoints in
`models/trained_v7fix/`. Heavy steps (training, feature extraction) are run from
the command line — see `run_all.sh`; this notebook works with their cached outputs.

In [ ]:
import sys, numpy as np
sys.path.insert(0, "../pipelines"); sys.path.insert(0, "..")
from _common import REPO, load_config, make_astra_config, load_cache

cfg = load_config()
ac  = make_astra_config(cfg)
print("classes:", ac.class_names)
print("ensemble spec:", cfg["ensemble"], "| weights:", cfg["weights_source"])

## 1. Consistent ensemble features

`pipelines/build_ensemble.py --spec v7fix6` extracts embeddings/logits for **every**
split with the same 6-model set (best+SWA × 3 seeds). Load the cached test split:

In [ ]:
te = load_cache(REPO / "cache" / cfg["ensemble"], "test")
print("test spectra:", te["emb"].shape, "| logits:", te["fine_logits"].shape)
pred = te["fine_logits"].argmax(1); y = np.asarray(te["labels"])
acc = (pred == y).mean()
per = [(pred[y==k]==k).mean() for k in range(len(ac.class_names))]
print(f"ensemble test accuracy: {acc:.3f} | macro recall: {np.mean(per):.3f}")

## 2. ASTRANet-Sentinel — anomaly detection (built on LUNA)

The [LUNA](https://github.com/<user>/LUNA) package is classifier-agnostic: it takes
plain arrays of embeddings + logits. Fit the 16 scorers on the train split and
score the 289-spectrum rare set:

In [ ]:
from luna import LUNAPipeline
from astranet.model import build_hierarchy_mappings

f2c, _, _ = build_hierarchy_mappings(ac.hierarchy, ac.class_names)
def luna_dict(f):
    o = {"embeddings": f["emb"].astype(np.float32), "logits": f["fine_logits"].astype(np.float32),
         "coarse_logits": f["coarse_logits"].astype(np.float32), "projections": f["proj_fine"].astype(np.float32)}
    if f.get("labels") is not None: o["labels"] = np.asarray(f["labels"])
    return o

tr = load_cache(REPO / "cache" / cfg["ensemble"], "train")
va = load_cache(REPO / "cache" / cfg["ensemble"], "val")
ra = load_cache(REPO / "cache" / cfg["ensemble"], "rare_unknown_combined")

pipe = LUNAPipeline(combiner="lightgbm", target_far=cfg["target_far"], fine_to_coarse=f2c)
pipe.fit(luna_dict(tr))
fam = pipe.family_analysis(luna_dict(va), luna_dict(ra))
for name, r in fam.items():
    print(f"{name:12s} AUROC={r['auroc']:.3f}  det {r['detected']}/{r['total']} @ {cfg['target_far']:.0%} FAR")

No single family suffices — the full Sentinel (LightGBM over all 16 scores, with
outlier-exposure fine-tuning under the leakage-free k-fold protocol of
`pipelines/oe_kfold.py`) is what reaches the headline detection rates. Its outputs
live in `results/oe/`:

In [ ]:
import json
final = json.load(open(REPO / "results/oe/oe_ensemble_v7fix_final.json"))
sa, un = final["score_avg"], final["union_or"]
print(f"3-seed score-average @1% FAR : {sa['per_spectrum']:.1%} of spectra, {sa['per_object']:.1%} of objects")
print(f"3-seed union-OR (~3% FAR)    : {un['per_spectrum']:.1%} of spectra, {un['per_object']:.1%} of objects")
print("tiers:", {k: f"{v['det']}/{v['n']}" for k, v in sa["tiers"].items()})

## 3. Conformal UQ (part of ASTRAnet)

APS prediction sets + Mondrian / AD-stratified conformal calibration live in
`astranet/uq/` and are driven by `pipelines/run_conformal.py`. Per-spectrum
p-values and coverage tables land in `results/<spec>/conformal/`:

In [ ]:
import pandas as pd
uq = pd.read_csv(REPO / f"results/{cfg['ensemble']}/conformal/uq_per_spectrum.csv")
uq.head()

## 4. Figures

All figures follow the paper standard in `luna.style` (final physical size, 9 pt
serif, no in-figure titles) with the ASTRAnet class palette in
`astranet.plotstyle`. Regenerate with:

```bash
python pipelines/make_umap.py --spec v7fix6
python pipelines/run_anomaly.py --spec v7fix6 --force   # detection/complementarity maps
```